<!-- ---
title: Element based 1D HideNN-FEM - L-BFGS training
format:
  html:
    html-math-method: mathjax
    code-fold: show
jupyter: python3
execute:
  echo: true
pdf-engine: pdflatex
--- -->

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "128"
os.environ["MKL_NUM_THREADS"] = "128"

# Element based 1D HideNN-FEM - L-BFGS training - with tensor decomposition

$\forall v\in V(\Omega),$ find $u\in H(\Omega)$,


$$\int_\Omega \nabla v \cdot \lambda(x) \nabla u = \int_\Omega f v  + \int_{\partial \Omega_N} g v$$

Avant on cherchait la solution sous la forme $u(x) = u_{model}(x)$.

On rajoute maintenant le module d'Young $E$ de la barre comme paramètre et on cherche la solution sous forme de décomposition tensorielle : 
$$u(x, E) = \sum_{i=1}^m \bar{u}_i(x) \lambda_i(E)$$

Avec chaque monômes $\bar{u}_i(x)$ ou $\lambda_i(E)$ étant un sous-modèle. (voir le schéma de Daby-Seesaram (2026)).

On garde la même discrétisation et le même degré d'interpolation pour les fonctions $\lambda_i$ que pour les modes spatiaux.

In [ ]:
#| code-fold: true

import torch
# At the top of your notebook
# torch.set_num_threads(128)
# print(f"Using {torch.get_num_threads()} CPU threads")  # should print 128

# Set device to cpu
device = torch.device("cpu")

import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook"
from IPython.display import clear_output

# # torch.set_default_dtype(torch.float32)
# print(torch.cuda.is_available())
# print(torch.cuda.get_device_name(0))

# # Set device
# device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

from pgd_plots import *

## Vectorised version of the Element-based implementation

We recode 1D shape functions in HideNN-FEM (first order).

A vectorised implementation enables batch processing of several points evaluation which in terns enables batch wise differentiation.

* In non-batched implementation

    * `du_dx = [torch.autograd.grad(u[i], x[i], grad_outputs=torch.ones_like(u[i]), create_graph=True) for i,_ in enumerate(u)`

* With the batched version
    * `du_dx = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u), create_graph=True) `


In [ ]:
class mySF1D_elementBased_vectorised(nn.Module):
    def __init__(self, connectivity):            
                                        # on doit maintenant l'init en lui donnant la connectivity table
        super(mySF1D_elementBased_vectorised, self).__init__()
        if connectivity.dim == 1:
            connectivity = connectivity[:,None] # il faut que la connectivity table soit toujours 2D
        self.connectivity = connectivity

        self.register_buffer('GaussPoint',self.GP()) # execute la fonction donc store les points de gauss
        self.register_buffer('w_g',torch.tensor(1.0))


    def UpdateConnectivity(self,connectivity):
        self.connectivity = connectivity.astype(int) # change le type seulement

    def GP(self):
        "Defines the position of the integration point(s) for the given element"

        return torch.tensor([[1/2, 1/2]], requires_grad=True)                                       # a1, a2, th 2 area coordinates

    def forward(self, 
                x               : torch.Tensor  = None  , 
                cell_id         : list          = None  , 
                coordinates     : torch.Tensor  = None  ,    # c est les points du mesh
                flag_training   : bool          = False):

        assert coordinates is not None, "No nodes coordinates provided. Aborting"

        cell_nodes_IDs  = self.connectivity[cell_id,:].T # id des noeus a partir de la connectivity et de l id de l element
        Ids             = torch.as_tensor(cell_nodes_IDs) .to(coordinates.device) .t() [:,:,None]     # si on etait en 2d juste ca rajoute un coord artificiellement pour etre en 3d 
        nodes_coord     = torch.gather(coordinates[:,None,:].repeat(1,2,1), 
                                       0, 
                                       Ids.repeat(1,1,1))    # [:,:,None] usefull only in 2+D  Ids.repeat(1,1,d) with d \in [1,3]
                                                             # so in 2D, repeat(1,1,2) ?
        
        nodes_coord = nodes_coord.to(self.GaussPoint.dtype)

        if flag_training: 
            refCoordg   = self.GaussPoint.repeat(cell_id.shape[0],1)    
            Ng          = refCoordg                                     #valeurs des shapes functions en l'unique point de Gauss de l'élément, mais comme on a un elem P1, on retombe sur les mêmes valeurs donc on confond les refCoordg et Ng ?
            x_g         = torch.einsum('enx,en->ex',nodes_coord,Ng)     #coordonnés des points de gauss dans le vrai mesh (il faudrait mettre refCoordg à la place de Ng si on voulait être plus général ?)
            refCoord    = self.GetRefCoord(x_g,nodes_coord)             #on retrouve les coordonnés des points de Gauss qui ont permis de calculer les x_g
            N           = refCoord                                      # QUESTION : on doit retrouver les valeurs de shapes functions qui ont permis de transporter les points de Gauss, mais c etait justement pas points de depart ?
            detJ        = nodes_coord[:,1] - nodes_coord[:,0]
            # print(N)
            return N, x_g, detJ*self.w_g

        else: 
            refCoord = self.GetRefCoord(x,nodes_coord)
            N = torch.stack((refCoord[:,0], refCoord[:,1]),dim=1)       # on calcule les coordonnes du point a evaluer dans le mesh de reference 
            return N




    def GetRefCoord(self,x, nodes_coord):
        ''''
        dfsgds
        '''
        InverseMapping          = torch.ones([int(nodes_coord.shape[0]), 2, 2], dtype=x.dtype, device=x.device)
        detJ                    = nodes_coord[:,0,0] - nodes_coord[:,1,0]
        InverseMapping[:,0,1]   = -nodes_coord[:,1,0]
        InverseMapping[:,1,1]   = nodes_coord[:,0,0]
        InverseMapping[:,1,0]   = -1*InverseMapping[:,1,0]
        InverseMapping[:,:,:]  /= detJ[:,None,None]
        x_extended = torch.stack((x, torch.ones_like(x)),dim=1)


        return torch.einsum('eij,ej...->ei',InverseMapping,x_extended.squeeze(1))

#### Recall on the iso-parametric Finite Element Method

In 1D, for P1 elements, there are two shape functions per element, $N_1\left(\xi\right)$ and $N_2\left(\xi\right)$, $\xi \in \left[0,1\right]$ being the coordinate in the reference element space.

The iso-parametric idea relies on using the same interpolation for the space coordinates as is used for the QoIs, which means that space is interpolated using the same shape functions as the displacement is for instance.
Thus, the real space coordinate $x$ satisfies
* $x = \sum_{i=1}^{2}N_i\left(\xi \right) x_i$, 

 with $x_i$ the coordinate of the node associated with the $i-$ th shape function.

 Such mapping can be expressed using the area coordinates $a_1$ and $a_2$ (such that $N_1\left(\xi\right) = a_1$ and $N_2\left(\xi\right) = a_2$).

$\begin{pmatrix}
x \\
1
\end{pmatrix} = \underbrace{\begin{bmatrix}
x_1 & x_2\\
1 & 1
\end{bmatrix}}_{\mathcal{M}} \begin{pmatrix}
a_1 \\
a_2
\end{pmatrix}.$

 Reciprocally (for non degenerated elements), 

$\begin{pmatrix}
a_1 \\
a_2
\end{pmatrix} = \underbrace{\frac{1}{x_1 - x_2}\begin{bmatrix}
1 & -x_2\\
-1 & x_1
\end{bmatrix}}_{\mathcal{M}^{-1}} \begin{pmatrix}
x \\
1
\end{pmatrix}.$

## Mesh generation

In [ ]:
N_u           = 17
nodes_u       = torch.linspace(0,6.28,N_u)
nodes_u       = nodes_u[:,None] # pq il l etend ?
elements_u    = torch.vstack([torch.arange(0,N_u-1),torch.arange(1,N_u)]).T

N_E = 25
nodes_E =   torch.linspace(1e2, 1e3, N_E)
nodes_E =   nodes_E[:,None]
elements_E    = torch.vstack([torch.arange(0,N_E-1),torch.arange(1,N_E)]).T

## Assembly using the vectorised element block

### Un monôme

In [ ]:
def inv_softplus(y):
    return torch.log(-torch.expm1(-y)) + y # numerically stable HMM ok.. merci claude

class interpolation1D(nn.Module):    
    def __init__(self, 
            nodes           : torch.Tensor  = None  , 
            elements        : list          = None  , 
            dirichlet       : list          = None  ,                 # Fixed nodes (by default, 2 extremities of the beam)
            n_components    : int           = 1     ,                    # Number of dofs per node 
            r_adaptivity : bool               = False
            ):                                 
        super().__init__()
        self.r_adaptivity = r_adaptivity

        if r_adaptivity:
            self.a = nodes[0][0].item()
            assert self.a == 0
            self.b = nodes[-1][0].item()
            self.n_nodes = nodes.numel()

            nodes_squeezed = nodes.squeeze().float()
            assert (nodes_squeezed[1:] > nodes_squeezed[:-1]).all(), \
                "Input nodes must be strictly increasing"

            # Differences between consecutive nodes — always > 0, safe for inv_softplus
            differences = nodes_squeezed[1:] - nodes_squeezed[:-1]  # (n_nodes-1,)
            increments = differences / (self.b - self.a)             # normalize, scale doesn't matter
            coordinates_init = inv_softplus(increments)              # (n_nodes-1,)

            # Sanity check: verify the inverse is correct
            with torch.no_grad():
                recovered_increments = F.softplus(coordinates_init)
                recovered_coords = torch.cumsum(recovered_increments, dim=0)
                recovered_coords = torch.cat([torch.zeros(1), recovered_coords])
                recovered_coords = self.a + (self.b - self.a) * (recovered_coords / recovered_coords[-1])
                assert torch.allclose(recovered_coords, nodes_squeezed, atol=1e-5), \
                    f"Inverse mapping failed: max error = {(recovered_coords - nodes_squeezed).abs().max():.2e}"
                print(f"Inverse mapping OK — max error: {(recovered_coords - nodes_squeezed).abs().max():.2e}")

            self.coordinates = nn.ParameterDict({
                'all': nn.Parameter(coordinates_init, requires_grad=False)
            })

        else:
            # Initialize with the fed mesh
            self.coordinates =nn.ParameterDict({                                            
                                        'all': nn.Parameter(nodes.float(),requires_grad = False)
                                        })
            
        self.n_components                       = n_components
        self.register_buffer('values',0.5*torch.ones((self.get_coordinates().shape[0], self.n_components)))        # QUESTION: a quoi il sert ce register_buffer
        self.dirichlet = dirichlet
        self.elements = elements

        self.Ne = len(elements)

        self.shape_functions = mySF1D_elementBased_vectorised(elements)              

        # To easily transfer to CUDA or change dtype of whole model
        self.register_buffer('one', torch.tensor([1], dtype=torch.float32))

        self.SetBCs()

    def SetBCs(self):
        assert self.n_components == 1, "only scalar field implemented. Aborting"
        if self.n_components == 1:
            self.dofs_free                  = (torch.ones_like(self.values[:])==1)[:,0] #la syntaxe de lenfer
            self.dofs_free[self.dirichlet]  = False
            
            nodal_values_imposed            = 0*self.values[~self.dofs_free,:]              # Not generic yet, only clamped-clamped BCs for now

            nodal_values_free               = self.values[self.dofs_free,:] 
            self.nodal_values               = nn.ParameterDict({
                                                'free'      : nodal_values_free,
                                                'imposed'   : nodal_values_imposed,
                                                })
            # Initialize to False and then we manage in the aggregate training function
            self.nodal_values['imposed'].requires_grad = False
            self.nodal_values['free'].requires_grad = False

    def get_coordinates(self):
        # We're training for the increments, so we compute the right increments to give an uniform mesh once reparametrized
        if self.r_adaptivity:
            # softplus ensures increments > 0 → coordinates strictly increasing
            increments = F.softplus(self.coordinates['all'])#.squeeze()        # shape: (n_nodes,)
            coords = torch.cumsum(increments, dim=0) #- increments       # sorted by construction
            coords = torch.cat((torch.tensor([self.a], device = coords.device), coords))
            # coords = torch.cat([torch.zeros(1), coords])

            # Normalize to fit in [a, b]
            coords =  self.a + (self.b - self.a) * (coords / coords[-1])
            return coords[:, None]
        else:
            # There's no reparametrization
            return self.coordinates['all']#.squeeze()

    def forward(self, x = None): 
        if self.training :
            k_elt = torch.arange(0,self.Ne)                                                            # QUESTION : on fait un forward pass qui va  exciter tout les neurones donc tous les elements ?     

        else :                                                                                 
            k_elt = []
            coords = self.get_coordinates()  # compute once
            for xx in x:
                found = False
                for k in range(self.Ne):
                    elt = self.elements[k]
                    if xx >= coords[elt[0]] and xx <= coords[elt[1]]:
                        k_elt.append(k)
                        found = True
                        break
                if not found:  # catches boundary floating point issues
                    k_elt.append(self.Ne - 1)


        if self.training : # train mode 
            shape_functions, x_g, detJ = self.shape_functions(
                x               = x                 , 
                cell_id         = k_elt             , 
                coordinates     = self.get_coordinates()        , 
                flag_training   = self.training     )
        else:   # eval mode 
            shape_functions = self.shape_functions(
                x               = x                 , 
                cell_id         = k_elt             , 
                coordinates     = self.get_coordinates()        , 
                flag_training   = self.training     )
            
        
        # Batch interpolation of the solution using the computed shape functions batch
        nodal_values_tensor                     = torch.ones_like(self.values) 
        nodal_values_tensor[self.dofs_free,:]   = self.nodal_values['free']
        nodal_values_tensor[~self.dofs_free,:]  = self.nodal_values['imposed']                    

        cell_nodes_IDs      = self.elements[k_elt,:].T
        Ids                 = torch.as_tensor(cell_nodes_IDs).to(nodal_values_tensor.device).t()[:,:,None]      # :,:,None] usefull only in 2+D  

        
        self.nodes_values   = torch.gather(nodal_values_tensor[:,None,:].repeat(1,2,1),
                                        0,
                                        Ids.repeat(1,1,1))    # [:,:,None] usefull only in 2+D  Ids.repeat(1,1,d) with d \in [1,3]
        self.nodes_values   = self.nodes_values.to(shape_functions.dtype)
        # shape_functions is N, th shape function values at the sole Gauss point in the element 
        u = torch.einsum('gi...,gi->g',self.nodes_values,shape_functions) 

        if self.training :
            return u, x_g, detJ
        else:
            return u


#### Proofing

In [ ]:
def square(x):
    return x*x

In [ ]:
# # nodes_u_non_uniform = square(nodes_u)/nodes_u[-1]
# # test = interpolation1D(nodes=nodes_u_non_uniform, elements=elements_u, dirichlet=[0, nodes_u.shape[0]-1], r_adaptivity=True)
# print("nodes ini :\n",nodes_u_non_uniform.squeeze(-1))
# # print(nodes_u.squeeze(-1).numel())
# coord = test.get_coordinates().squeeze(-1)
# print("after the remap :\n",coord)
# # print(coord.numel())


In [ ]:
# print(list(test.named_parameters()))

In [ ]:
# plt.scatter(nodes_u_non_uniform.numpy(), np.zeros_like(nodes_u.numpy()))

### La PGD - somme des modes

Pour la r_adaptivity, on n'entraîne le maillage qu'à partir du second mode.
Pour laisser cette r-adaptivity comme un outil de raffinage, pas direct. EST-CE PERTINENT ?

In [ ]:
class  PGDapprox(nn.Module):
    def __init__(self,
                 u_nodes          : torch.Tensor = None,
                 E_nodes          : torch.Tensor = None,
                 elements_u       : list         = None,
                 elements_E       : list         = None,
                 max_mode_index : int          = 1,
                 r_adaptivity :bool            = False,
                 train_first_mesh = False,
                 shared_mesh = True):
        
        super().__init__()

        self.max_mode_index = max_mode_index

        dirichlet = [0 ,nodes_u.shape[0]-1]#, nodes_u.shape[0]//2
        self.u_modes = nn.ModuleList(
            [interpolation1D(nodes= u_nodes, elements=elements_u, dirichlet=dirichlet, r_adaptivity=train_first_mesh)] + \
            [interpolation1D(nodes= u_nodes, elements=elements_u, dirichlet=dirichlet, r_adaptivity=r_adaptivity) for _ in range(max_mode_index-1)] #nodes_u.shape[0]-1
        )

        # Force all modes to share the same coordinates parameter
        if r_adaptivity and shared_mesh and (max_mode_index > 1):
            
            if train_first_mesh: start = 0
            else: start = 1

            shared_coords = self.u_modes[start].coordinates['all']  # take from first r_adaptivity mode
            for mode in self.u_modes[start+1:]:                        # assign to all others
                mode.coordinates['all'] = shared_coords


        self.E_modes = nn.ModuleList(
            [interpolation1D(nodes=E_nodes, elements=elements_E, dirichlet=[]) for _ in range(max_mode_index)]
        )
        
        self.current_index = 0

    def forward(self, x=None, E=None):
        if self.training:
            results_u = [self.u_modes[i]() for i in range(self.current_index)]
            results_E = [self.E_modes[i]() for i in range(self.current_index)]

            # Unpack — each u_vals[i] retains its graph to x_gs[i]
            u_vals = [r[0] for r in results_u]   # list of (Ne,) tensors, graphs intact
            x_gs   = [r[1] for r in results_u]   # list of (Ne,) Gauss point tensors
            J_us   = [r[2] for r in results_u]

            lmbda_vals = [r[0] for r in results_E]
            E_gs   = [r[1] for r in results_E]
            J_Es   = [r[2] for r in results_E]

            coordinates_nodes_u = [self.u_modes[i].get_coordinates() for i in range(self.current_index)]

            return u_vals, lmbda_vals, x_gs, E_gs, J_us, J_Es, coordinates_nodes_u

        else:  # inference: Σ u_i(x) · E_i(E)
            # I want to return as much displacement field as there is E values
            results = []
            for E_val in E:
                u_val = sum(self.u_modes[i](x) * self.E_modes[i](E_val.unsqueeze(0)) for i in range(self.current_index))
                results.append(u_val)
            return results

In [ ]:
max_num_mode = 3
model_ex = PGDapprox(u_nodes=nodes_u, E_nodes=nodes_E, elements_u=elements_u, elements_E=elements_E, max_mode_index=max_num_mode, r_adaptivity=True, shared_mesh=False)

print(model_ex.u_modes[1].coordinates['all'].data_ptr())
print(model_ex.u_modes[2].coordinates['all'].data_ptr())  # should print True

In [ ]:
# print(list(model_ex.named_parameters()))

trainable     = [(n, p) for n, p in model_ex.named_parameters() if p.requires_grad]
frozen        = [(n, p) for n, p in model_ex.named_parameters() if not p.requires_grad]

print(f"Trainable parameters:  {len(trainable)}")
print(f"Frozen parameters:     {len(frozen)}")

## Training with batch version




### Grid intersection

In [ ]:
def intersect_grids(i_grid, j_grid, tol=1e-10):
    # Force 1D regardless of input shape
    i_grid = i_grid.reshape(-1)
    j_grid = j_grid.reshape(-1)

    # 1. Union + sort
    nodes, _ = torch.sort(torch.cat([i_grid, j_grid]))

    # # 2. Remove near-duplicates
    # gaps = torch.diff(merged)
    # keep = torch.cat([torch.tensor([True], device=merged.device), gaps > tol])
    # nodes = merged[keep]

    # # 3. Element quantities
    h         = torch.diff(nodes)
    gauss_pts = nodes[:-1] + 0.5 * h
    det_J     = 0.5 * h

    return det_J, gauss_pts

### Energy

In [ ]:
def PotentialEnergy_PGD(u_vals, lmbda_vals, x_gs, E_gs, J_us, J_Es, coordinates_nodes_u, model, f, fault_amp):
    def _debug_stack(name, x):
        if isinstance(x, list):
            x = torch.stack(x)
        # print(f"{name}.shape = {x.shape}")
        return x

    du_dx = []
    for u, x_g in zip(u_vals, x_gs):

        du_dx_i = torch.autograd.grad(
            u,
            x_g,
            grad_outputs=torch.ones_like(u),
            create_graph=True
        )[0]

        du_dx.append(du_dx_i)
    du_dx = torch.stack(du_dx)
    
    
    lmbda_vals = _debug_stack("lmbda_vals", lmbda_vals)
    u_vals = _debug_stack("u_vals", u_vals)
    x_gs   = _debug_stack("x_gs", x_gs)
    J_us   = _debug_stack("J_us", J_us)
    E_gs   = _debug_stack("E_gs", E_gs)
    J_Es   = _debug_stack("J_Es", J_Es)
    du_dx = _debug_stack('du_dx', du_dx)
    # coordinates_nodes_u = _debug_stack('coordinates_nodes_u', coordinates_nodes_u)
    # Experimental
    J_Es  = J_Es.squeeze(-1)[0] 
    J_us  = J_us.squeeze(-1)
    E_gs  = E_gs.squeeze(-1)
    x_gs  = x_gs.squeeze(-1)
    du_dx = du_dx.squeeze(-1)
    # coordinates_nodes_u = coordinates_nodes_u.squeeze(-1)

    lmbda_vals = _debug_stack("lmbda_vals", lmbda_vals)
    u_vals = _debug_stack("u_vals", u_vals)
    x_gs   = _debug_stack("x_gs", x_gs)
    J_us   = _debug_stack("J_us", J_us)
    E_gs   = _debug_stack("E_gs", E_gs)
    J_Es   = _debug_stack("J_Es", J_Es)
    # coordinates_nodes_u = _debug_stack('coordinates_nodes_u', coordinates_nodes_u)
    # print(J_Es)

    
    ## gradient term
    # grad_term = torch.einsum('ie,je,e,ixk,jxk,x->', lmbda_vals, lmbda_vals, J_Es, )
    E_g = E_gs[0]

    ## Le terme spatial est plus compliqué due au différentes grids.
    # Diag term
    param_term_diag = torch.einsum('ne,ne,e,e->n', lmbda_vals, lmbda_vals, E_g, J_Es)
    space_term_diag = torch.einsum('nx,nx,nx->n', du_dx, du_dx, J_us) # lui est correct
    grad_term_diag_term = torch.einsum('n,n->', space_term_diag, param_term_diag)

    # Cross term
    param_term_whole = torch.einsum('me,ne,e,e->mn', lmbda_vals, lmbda_vals, E_g, J_Es)

    n_mode = u_vals.shape[0]
    cross_terms = []
    for i in range(n_mode):
        for j in range(i):
            i_grid, j_grid = coordinates_nodes_u[i], coordinates_nodes_u[j]
            J_u_intersect, gauss_points_intersect = intersect_grids(i_grid, j_grid, tol = 1e-7)
            gauss_points_intersect = gauss_points_intersect.detach().requires_grad_(True)  # ← this must be here

            model.u_modes[i].eval()
            model.u_modes[j].eval()

            # I need to go evaluate the i and j modes at those new gauss points
            u_i = model.u_modes[i](gauss_points_intersect)
            u_j =  model.u_modes[j](gauss_points_intersect)

            du_i_dx = torch.autograd.grad(u_i, gauss_points_intersect,
                                            grad_outputs=torch.ones_like(u_i),
                                            create_graph=True#create_graph_list[i]
                                            )[0]
            du_j_dx = torch.autograd.grad(u_j, gauss_points_intersect,
                                            grad_outputs=torch.ones_like(u_j),
                                            create_graph= True#create_graph_list[j]
                                            )[0]
            
            model.u_modes[i].train()
            model.u_modes[j].train()

            space_term_ij = (du_i_dx*du_j_dx*J_u_intersect).sum()
            cross_terms.append(space_term_ij*param_term_whole[i,j])
            
    
    grad_term_triangular_term = (2 * torch.stack(cross_terms).sum() if cross_terms else torch.tensor(0.0, device=du_dx.device, dtype=du_dx.dtype))
    grad_term = 2*grad_term_triangular_term + grad_term_diag_term

    ## volumic term
    f_tensor = f(x_gs, fault_amp)
    f_tensor  = f_tensor.squeeze(-1) 
    f_tensor   = _debug_stack("f_tensor", f_tensor)

    space_term_vol = torch.einsum('kx,kx,kx->k', f_tensor, u_vals, J_us)
    param_term_vol = torch.einsum('ke,e->k', lmbda_vals, J_Es)

    volumic_term = torch.einsum('k,k->', space_term_vol, param_term_vol)
        
    # for k in range(len(u_vals)):
    #     E_term_volumic = torch.sum(lmbda_vals[k]*J_Es)
    #     u_term_volumic = torch.sum(u_vals[k]*f_tensor*J_us)

    #     volumic_term += E_term_volumic*u_term_volumic
        
    energy    =  0.5*grad_term - volumic_term

    return energy

## force volumique (le poids par exemple)
def f(x, fault_amp):
    # x: shape [k, d] or [d]
    
    if x.dim() == 1:
        fault = torch.zeros_like(x)
        fault[x.numel() // 4] = 1
        return 1000 * (torch.ones_like(x) + fault)

    # x is [k, d]
    k, d = x.shape
    fault = torch.zeros_like(x)

    idx = d // 4
    fault[:, idx] = fault_amp  # apply per row

    return 1000 * (torch.ones_like(x) + fault)

### Aggregate training function

In [ ]:
def train_PGD(
    nodes_u=nodes_u, nodes_E=nodes_E,
    elements_u=elements_u, elements_E=elements_E,
    mode_by_mode_nodal_values=False,
    max_num_mode=5, r_adaptivity=False, mode_by_mode_r_adapt=True, shared_mesh=True, train_first_mesh=False,
    Nepoch=[500, 500, 500, 500, 500], save_plot_rate=500,
    lr=0.1, eta=1e-5, optimizer_type='Adam',
    results=None,
    f=f, fault_amp=0,
    log_parameters=False
):
    assert results is not None
    # if optimizer_type not in ['Adam', 'LBFGS', 'AdamW']:
    #     raise ValueError(f"Unknown optimizer: {optimizer_type}")

    model = PGDapprox(
        u_nodes=nodes_u.clone(), E_nodes=nodes_E.clone(),
        elements_u=elements_u.clone(), elements_E=elements_E.clone(),
        max_mode_index=max_num_mode,
        r_adaptivity=r_adaptivity, shared_mesh=shared_mesh, train_first_mesh=train_first_mesh
    ).to(device)

    if not isinstance(lr, list):
        lr = max_num_mode * [lr]
    if not isinstance(optimizer_type, list):
        optimizer_type = max_num_mode * [optimizer_type]

    results["model"] = model
    model.train()
    lossLists = []

    for k in range(max_num_mode):
        model.current_index += 1
        idx = model.current_index  # 1-based
        print(f"\nAdding mode {idx}")

        ######
        # Si r adaptivity
        if r_adaptivity:
            # Si on est au premier mode, on suit la consigne
            if model.current_index == 1:
                 model.u_modes[0].coordinates['all'].requires_grad = train_first_mesh

            else :     
                # Si on entraîne un par un, on désactive tous les grads puis on active que le courant
                if mode_by_mode_r_adapt:
                    for i in range(model.current_index):
                        model.u_modes[i].coordinates['all'].requires_grad = False
                    model.u_modes[model.current_index - 1].coordinates['all'].requires_grad = True
                # Sinon on active tous les grads
                else :
                     for i in range(model.current_index):
                        model.u_modes[i].coordinates['all'].requires_grad = True

        #######
        # On gére l'adaptation des valeurs nodales
        # On entraîne tout le temps le premier mode
        
        # Si on est au premier mode, on adapte toujours
        if model.current_index == 1:
                model.u_modes[0].nodal_values.free.requires_grad = True
                model.E_modes[0].nodal_values.free.requires_grad = True
        else :     
            # Si on entraîne un par un, on désactive tous les grads puis on active que le courant
            if mode_by_mode_nodal_values:
                for i in range(model.current_index):
                    model.u_modes[i].nodal_values.free.requires_grad = False
                    model.E_modes[i].nodal_values.free.requires_grad = False
                
                model.u_modes[model.current_index - 1].nodal_values.free.requires_grad = True
                model.E_modes[model.current_index - 1].nodal_values.free.requires_grad = True
            # Sinon on active tous les grads
            else:
                # print(range(model.current_index))
                for i in range(model.current_index):
                    model.u_modes[i].nodal_values.free.requires_grad = True
                    model.E_modes[i].nodal_values.free.requires_grad = True

        if log_parameters:
            trainable = [name for name, p in model.named_parameters() if p.requires_grad]
            print(f"############# TRAINABLE PARAMETERS for mode {idx}")
            print(trainable)

        # --- store initial Gauss points from first mode ---
        if idx == 1:
            _, _, L_x_g_ini, _, _, _, _ = model()
            results['x_gauss_ini'] = L_x_g_ini[0].squeeze(-1).detach()

        # --- optimizer ---
        trainable_params = filter(lambda p: p.requires_grad, model.parameters())
        if optimizer_type[k] == 'Adam':
            optimizer = torch.optim.Adam(trainable_params, lr=lr[k])
        elif optimizer_type[k] == 'AdamW':
            optimizer = torch.optim.AdamW(trainable_params, lr=lr[k], weight_decay=1e-2)
        else:
            optimizer = torch.optim.LBFGS(trainable_params, line_search_fn="strong_wolfe")

        def closure():
            optimizer.zero_grad()
            L_u, L_lmbda, L_x_g, L_E_g, L_J_u, L_J_E, L_coords = model()
            loss = PotentialEnergy_PGD(L_u, L_lmbda, L_x_g, L_E_g, L_J_u, L_J_E, L_coords, model, f, fault_amp)
            loss.backward()
            return loss

        # --- training loop ---
        n_epochs = Nepoch[k] if k < len(Nepoch) else Nepoch[-1]
        lossTraining = []

        for i in range(n_epochs):
            optimizer.step(closure)
            loss = closure()
            lossTraining.append(loss.item())

            if (i % save_plot_rate == 0) and not log_parameters:
                clear_output(wait=True)
                plt.figure(figsize=(6, 4))
                plt.plot(lossTraining)
                plt.title(f"Mode {idx} — Epoch {i}/{n_epochs}")
                plt.xlabel("Epoch")
                plt.ylabel("Loss")
                plt.grid(True)
                plt.tight_layout()
                plt.show()
                results[f'model_{i + n_epochs}'] = model
                results['model']     = model
                results['lossLists'] = lossLists

        lossLists.append(lossTraining)
        print(f"{i = } | loss = {loss.data:.2e}")

        # --- stagnation check ---
        if len(lossLists) > 1:
            stagnation_indic = abs(
                2 * (lossLists[-2][-1] - lossLists[-1][-1]) /
                    (lossLists[-2][-1] + lossLists[-1][-1])
            )
            print(f"stagnation indicator = {stagnation_indic:.6e}")
            if stagnation_indic < eta:
                print("Stopping due to stagnation")
                break

    results['model']     = model
    results['lossLists'] = lossLists

In [ ]:
x = torch.linspace(0,5, 20)
y = f(x, fault_amp=1)

fig = go.Figure()
fig.add_trace(go.Scatter( x = x, y=y, mode='lines+markers', name="Non-uniform load", line=dict(color='#01426a')))
fig.update_layout(
    margin=dict(l=0, r=0, t=0, b=0),
    plot_bgcolor='rgba(0,0,0,0)',  # Remove background color
    width=350, 
    height=200,  
    xaxis=dict(title='x',
    showgrid=True,
    gridcolor='lightgray'),
    yaxis=dict(title='f',
    tickvals=[-4.022e8, -4.016e8, -4.008e8],
    ticktext=['-4.022e8', '-4.016e8', '-4.008e8'],
    showgrid=True,
    gridcolor='lightgray',),
)

fig.show()

# Test runs

## no r-adapt + nodal values one by one + meshes one by one + different meshes

### Training and definition

In [ ]:
N_u           = 25
nodes_u       = torch.linspace(0,6.28,N_u)
nodes_u       = nodes_u[:,None] # pq il l etend ?
elements_u    = torch.vstack([torch.arange(0,N_u-1),torch.arange(1,N_u)]).T

N_E = 25
nodes_E =   torch.linspace(1e2, 1e3, N_E)
nodes_E =   nodes_E[:,None]
elements_E    = torch.vstack([torch.arange(0,N_E-1),torch.arange(1,N_E)]).T

In [ ]:
results_ref = {}
max_num_mode_ref = 3

train_PGD(
    nodes_u=nodes_u,
    elements_u=elements_u,
    nodes_E=nodes_E,
    elements_E=elements_E,
    max_num_mode    = max_num_mode_ref,
    Nepoch          = [200, 200, 500],
    save_plot_rate       = 50,
    lr              = [1e-1, 1e-2, 1e-2],
    r_adaptivity              = False,
    mode_by_mode_nodal_values = False,
    # mode_by_mode_r_adapt      = True,
    # shared_mesh     = False,
    # train_first_mesh= False,
    eta             = 1e-8,
    optimizer_type='Adam',
    fault_amp       = 0,           # On ajoute de l'asymétrie
    results = results_ref,
    # log_parameters=True
)

In [ ]:
## Unpacking

model_ref = results_ref['model'] 
lossLists = results_ref['lossLists']
x_gauss_ini = results_ref['x_gauss_ini']

## Saving
# torch.save(model.state_dict(), "model_weights_4.pt")

### Total loss

In [ ]:
plot_loss(lossLists)

### Different modes

In [ ]:
plot_modes(model_ref)

In [ ]:
model_ref.eval()

x_test = torch.linspace(0, 6.28, 50)

# ======================
# PARAMÈTRES MANUELS
# ======================
E1 = 110.0
E2 = 900.0

# --- run 1 ---
E_test = torch.tensor([E1, E2])
plot_displacement_field(model_ref, E_test)

### Midpoint deflection versus $E$

In [ ]:
plot_midpoint_vs_E(model_ref)

## r-adapt + nodal values one by one + meshes one by one + different meshes

### Training and definition

In [ ]:
N_u           = 25
nodes_u       = torch.linspace(0,6.28,N_u)
nodes_u       = nodes_u[:,None] # pq il l etend ?
nodes_u_non_uniform = square(nodes_u)/nodes_u[-1]
elements_u    = torch.vstack([torch.arange(0,N_u-1),torch.arange(1,N_u)]).T

N_E = 25
nodes_E =   torch.linspace(1e2, 1e3, N_E)
nodes_E =   nodes_E[:,None]
elements_E    = torch.vstack([torch.arange(0,N_E-1),torch.arange(1,N_E)]).T

In [ ]:
results_1 = {}
max_num_mode_1 = 5

train_PGD(
    nodes_u=nodes_u_non_uniform,
    elements_u=elements_u,
    nodes_E=nodes_E,
    elements_E=elements_E,
    max_num_mode    = max_num_mode_1,
    Nepoch          = [500, 500, 500, 500, 500],
    save_plot_rate       = 100,
    lr              = [1e-2, 1e-3, 1e-3, 1e-3, 1e-4],
    r_adaptivity    = True,
    mode_by_mode_nodal_values = True,
    mode_by_mode_r_adapt      = True,
    shared_mesh     = False,
    train_first_mesh= True,
    eta             = 1e-8,
    optimizer_type='Adam',
    fault_amp       = 0,           # On ajoute de l'asymétrie
    results = results_1,
    # log_parameters=True
)

In [ ]:
## Unpacking

model = results_1['model']
lossLists = results_1['lossLists']
x_gauss_ini = results_1['x_gauss_ini']

## Saving
# torch.save(model.state_dict(), "model_weights_4.pt")

### Total loss

In [ ]:
plot_loss(lossLists)

### Nodes displacement (only for r - adaptivity)

In [ ]:
plot_node_displacement(model, nodes_u_non_uniform)

### Different modes

In [ ]:
plot_modes(model)

### Displacement of the Gauss point (trick)

In [ ]:
plot_gauss_displacement(model, x_gauss_ini)

On plotte $u$ aux points de Gauss en fin de training (qui sont sont donc indicateurs de comment on été déplacés les noeuds du maillage) et aux points de Gauss initiaux (sauvegardés suite à l'initialisation).

In [ ]:
model.eval()

x_test = torch.linspace(0, 6.28, 50)

# ======================
# PARAMÈTRES MANUELS
# ======================
E1 = 110.0
E2 = 900.0

# --- run 1 ---
E_test = torch.tensor([E1, E2])
plot_displacement_field(model, E_test)

### Midpoint deflection versus $E$

In [ ]:
plot_midpoint_vs_E(model)

### Compared to the ref solution

In [ ]:
compute_error_vs_ref(model, model_ref, plot=True)

## r-adapt + nodal values all at once + meshes one by one + different meshes

### Training and definition

In [ ]:
N_u           = 25
nodes_u       = torch.linspace(0,6.28,N_u)
nodes_u       = nodes_u[:,None] # pq il l etend ?
nodes_u_non_uniform = square(nodes_u)/nodes_u[-1]
elements_u    = torch.vstack([torch.arange(0,N_u-1),torch.arange(1,N_u)]).T

N_E = 25
nodes_E =   torch.linspace(1e2, 1e3, N_E)
nodes_E =   nodes_E[:,None]
elements_E    = torch.vstack([torch.arange(0,N_E-1),torch.arange(1,N_E)]).T

In [ ]:
results_2 = {}
max_num_mode_2 = 5

train_PGD(
    nodes_u=nodes_u_non_uniform,
    elements_u=elements_u,
    nodes_E=nodes_E,
    elements_E=elements_E,
    max_num_mode    = max_num_mode_2,
    Nepoch          = [100, 100, 100, 100, 100],
    save_plot_rate       = 25,
    lr              = [1e-2, 1e-2, 1e-2, 1e-2, 1e-2],
    r_adaptivity    = True,
    mode_by_mode_nodal_values = False,
    mode_by_mode_r_adapt      = True,
    shared_mesh     = False,
    train_first_mesh= True,
    eta             = 1e-8,
    optimizer_type='Adam',
    fault_amp       = 0,           # On ajoute de l'asymétrie
    results = results_2,
    # log_parameters=True
)

In [ ]:
## Unpacking

model = results_2['model']
lossLists = results_2['lossLists']
x_gauss_ini = results_2['x_gauss_ini']

## Saving
# torch.save(model.state_dict(), "model_weights_4.pt")

### Total loss

In [ ]:
plot_loss(lossLists)

### Nodes displacement (only for r - adaptivity)

In [ ]:
plot_node_displacement(model, nodes_u_non_uniform)

### Different modes

In [ ]:
plot_modes(model)

### Displacement of the Gauss point (trick)

In [ ]:
plot_gauss_displacement(model, x_gauss_ini)

On plotte $u$ aux points de Gauss en fin de training (qui sont sont donc indicateurs de comment on été déplacés les noeuds du maillage) et aux points de Gauss initiaux (sauvegardés suite à l'initialisation).

In [ ]:
model.eval()

x_test = torch.linspace(0, 6.28, 50)

# ======================
# PARAMÈTRES MANUELS
# ======================
E1 = 110.0
E2 = 900.0

# --- run 1 ---
E_test = torch.tensor([E1, E2])
plot_displacement_field(model, E_test)

### Midpoint deflection versus $E$

In [ ]:
plot_midpoint_vs_E(model)

### Compared to the ref solution

In [ ]:
compute_error_vs_ref(model, model_ref, plot=True)

## r-adapt + nodal values one by one + meshes all + different meshes

### Training and definition

In [ ]:
N_u           = 25
nodes_u       = torch.linspace(0,6.28,N_u)
nodes_u       = nodes_u[:,None] # pq il l etend ?
nodes_u_non_uniform = square(nodes_u)/nodes_u[-1]
elements_u    = torch.vstack([torch.arange(0,N_u-1),torch.arange(1,N_u)]).T

N_E = 25
nodes_E =   torch.linspace(1e2, 1e3, N_E)
nodes_E =   nodes_E[:,None]
elements_E    = torch.vstack([torch.arange(0,N_E-1),torch.arange(1,N_E)]).T

In [ ]:
results_3 = {}
max_num_mode_3 = 5

train_PGD(
    nodes_u=nodes_u_non_uniform,
    elements_u=elements_u,
    nodes_E=nodes_E,
    elements_E=elements_E,
    max_num_mode    = max_num_mode_3,
    Nepoch          = [100, 250, 250, 250, 250],
    save_plot_rate       = 50,
    lr              = [1e-2, 1e-2, 1e-2, 1e-2, 1e-2],
    r_adaptivity    = True,
    mode_by_mode_nodal_values = True,
    mode_by_mode_r_adapt      = False,
    shared_mesh     = False,
    train_first_mesh= True,
    eta             = 1e-8,
    optimizer_type='Adam',
    fault_amp       = 0,           # On ajoute de l'asymétrie
    results = results_3,
    # log_parameters=True
)

In [ ]:
## Unpacking

model = results_3['model']
lossLists = results_3['lossLists']
x_gauss_ini = results_3['x_gauss_ini']

## Saving
# torch.save(model.state_dict(), "model_weights_4.pt")

### Total loss

In [ ]:
plot_loss(lossLists)

### Nodes displacement (only for r - adaptivity)

In [ ]:
plot_node_displacement(model, nodes_u_non_uniform)

### Different modes

In [ ]:
plot_modes(model)

### Displacement of the Gauss point (trick)

In [ ]:
plot_gauss_displacement(model, x_gauss_ini)

On plotte $u$ aux points de Gauss en fin de training (qui sont sont donc indicateurs de comment on été déplacés les noeuds du maillage) et aux points de Gauss initiaux (sauvegardés suite à l'initialisation).

In [ ]:
model.eval()

x_test = torch.linspace(0, 6.28, 50)

# ======================
# PARAMÈTRES MANUELS
# ======================
E1 = 110.0
E2 = 900.0

# --- run 1 ---
E_test = torch.tensor([E1, E2])
plot_displacement_field(model, E_test)

### Midpoint deflection versus $E$

In [ ]:
plot_midpoint_vs_E(model)

### Compared to the ref solution

In [ ]:
compute_error_vs_ref(model, model_ref, plot=True)

## r-adapt + nodal values all + meshes all + different meshes

### Training and definition

In [ ]:
N_u           = 25
nodes_u       = torch.linspace(0,6.28,N_u)
nodes_u       = nodes_u[:,None] # pq il l etend ?
nodes_u_non_uniform = square(nodes_u)/nodes_u[-1]
elements_u    = torch.vstack([torch.arange(0,N_u-1),torch.arange(1,N_u)]).T

N_E = 25
nodes_E =   torch.linspace(1e2, 1e3, N_E)
nodes_E =   nodes_E[:,None]
elements_E    = torch.vstack([torch.arange(0,N_E-1),torch.arange(1,N_E)]).T

In [ ]:
results_4 = {}
max_num_mode_4 = 4

# train_PGD(
#     nodes_u=nodes_u,
#     elements_u=elements_u,
#     nodes_E=nodes_E,
#     elements_E=elements_E,
#     max_num_mode    = max_num_mode_4,
#     Nepoch          = [300, 500, 250, 250, 250],
#     save_plot_rate       = 25,
#     lr              = [1e-2, 1e-2, 1e-3, 1e-3, 1e-3],
#     r_adaptivity    = True,
#     mode_by_mode_nodal_values = False,
#     mode_by_mode_r_adapt      = False,
#     shared_mesh     = False,
#     train_first_mesh= True,
#     eta             = 1e-8,
#     optimizer_type='Adam',
#     fault_amp       = 0,           # On ajoute de l'asymétrie
#     results = results_4,
#     # log_parameters=True
# )

train_PGD(
    nodes_u=nodes_u_non_uniform,
    elements_u=elements_u,
    nodes_E=nodes_E,
    elements_E=elements_E,
    max_num_mode    = max_num_mode_4,
    Nepoch          = [250, 250, 500, 1000],
    save_plot_rate       = 25,
    lr              = [1e-2, 1e-2, 1e-3, 1e-3, 1e-3], #[1e-2, 1e-2, 1e-3, 1e-4, 1e-4] diverges [1e-2, 1e-3, 1e-4, 1e-4, 1e-4]
    r_adaptivity    = True,
    mode_by_mode_nodal_values = False,
    mode_by_mode_r_adapt      = False,
    shared_mesh     = False,
    train_first_mesh= True,
    eta             = 1e-6,
    optimizer_type=['Adam', 'Adam', 'Adam', 'Adam', 'Adam'],
    fault_amp       = 0,           # On ajoute de l'asymétrie
    results = results_4,
    # log_parameters=True
)

In [ ]:
## Unpacking

model = results_4['model']
lossLists = results_4['lossLists']
x_gauss_ini = results_4['x_gauss_ini']

## Saving
# torch.save(model.state_dict(), "model_weights_4.pt")

### Total loss

In [ ]:
plot_loss(lossLists)

### Nodes displacement (only for r - adaptivity)

In [ ]:
plot_node_displacement(model, nodes_u_non_uniform)

### Different modes

In [ ]:
plot_modes(model)

### Displacement of the Gauss point (trick)

In [ ]:
plot_gauss_displacement(model, x_gauss_ini)

On plotte $u$ aux points de Gauss en fin de training (qui sont sont donc indicateurs de comment on été déplacés les noeuds du maillage) et aux points de Gauss initiaux (sauvegardés suite à l'initialisation).

In [ ]:
model.eval()

x_test = torch.linspace(0, 6.28, 50)

# ======================
# PARAMÈTRES MANUELS
# ======================
E1 = 110.0
E2 = 900.0

# --- run 1 ---
E_test = torch.tensor([E1, E2])
plot_displacement_field(model, E_test)

### Midpoint deflection versus $E$

In [ ]:
plot_midpoint_vs_E(model)

### Compared to the ref solution

In [ ]:
compute_error_vs_ref(model, model_ref, plot=True)